# Deep Card Hand Recognizer
## Regularized Neural Network — Built from First Principles


### Section 1 — Core Math: Activations & Loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Activation Functions ───────────────────────────────────────────────────────

def rectify(z):
    return np.maximum(0.0, z)

def rectify_prime(z):
    return (z > 0).astype(np.float32)

def logistic(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def logistic_prime(z):
    s = logistic(z)
    return s * (1.0 - s)

def stable_softmax(logit_matrix):
    shifted_logits = logit_matrix - logit_matrix.max(axis=1, keepdims=True)
    exp_l = np.exp(shifted_logits)
    return exp_l / exp_l.sum(axis=1, keepdims=True)

In [ ]:
# ── Loss: Class-Weighted Cross Entropy ────────────────────────────────────────

def weighted_cross_entropy(prob_out, targets, class_wts):
    """
    Weighted cross-entropy loss.
    prob_out  : (N, C) softmax output
    targets   : (N,)  integer class labels
    class_wts : (C,)  per-class weights
    """
    N = targets.shape[0]
    safe_probs = np.clip(prob_out, 1e-15, 1.0 - 1e-15)
    log_probs  = -np.log(safe_probs[np.arange(N), targets])
    sample_wts = class_wts[targets]          # weight for each sample's true class
    return np.sum(sample_wts * log_probs) / N

def weighted_cross_entropy_grad(prob_out, targets, class_wts):
    """
    Gradient of weighted cross-entropy w.r.t. softmax input.
    Multiply the standard (softmax - one_hot) grad by per-sample weights.
    """
    N = targets.shape[0]
    grad_mat = prob_out.copy()
    grad_mat[np.arange(N), targets] -= 1.0
    sample_wts = class_wts[targets].reshape(-1, 1)
    return (grad_mat * sample_wts) / N

def compute_class_weights(label_array, num_classes):
    """
    Compute inverse-frequency class weights from training labels.
    Returns array of shape (num_classes,).
    """
    counts = np.bincount(label_array, minlength=num_classes).astype(float)
    counts = np.where(counts == 0, 1, counts)   # avoid 0 for absent classes
    wts = label_array.shape[0] / (num_classes * counts)
    return wts

### Section 2 — Building Blocks: Layers

In [ ]:
# ── Linear (Dense) Layer with L2 Regularization ───────────────────────────────

class LinearUnit:
    """
    Fully connected linear layer with optional L2 weight decay.
    W initialized with He scaling: std = sqrt(2 / fan_in).
    """
    def __init__(self, in_dim, out_dim, l2_lambda=0.0):
        self.W  = np.random.randn(in_dim, out_dim).astype(np.float32) * np.sqrt(2.0 / in_dim)
        self.bv = np.zeros((1, out_dim), dtype=np.float32)
        self.lam = l2_lambda     # L2 strength
        self._inp = None
        self.raw_out = None

    def forward(self, inp):
        self._inp   = inp
        self.raw_out = inp @ self.W + self.bv
        return self.raw_out

    def backward(self, grad_from_above, alpha):
        dW = self._inp.T @ grad_from_above
        db = grad_from_above.sum(axis=0, keepdims=True)
        d_in = grad_from_above @ self.W.T

        # L2 penalty term added to weight gradient
        dW += self.lam * self.W

        self.W  -= alpha * dW
        self.bv -= alpha * db
        return d_in

In [ ]:
# ── Batch Normalization Layer ──────────────────────────────────────────────────

class BatchNorm:
    def __init__(self, num_features, momentum=0.9, eps=1e-5):
        self.gamma    = np.ones((1, num_features), dtype=np.float32)   # scale
        self.beta     = np.zeros((1, num_features), dtype=np.float32)  # shift
        self.momentum = momentum
        self.eps      = eps

        # Running statistics for inference
        self.running_mean = np.zeros((1, num_features), dtype=np.float32)
        self.running_var  = np.ones((1, num_features),  dtype=np.float32)

        # Cache for backward pass
        self._cache = {}

    def forward(self, z, training=True):
        if training:
            mu  = z.mean(axis=0, keepdims=True)
            var = z.var(axis=0,  keepdims=True)
            z_hat = (z - mu) / np.sqrt(var + self.eps)

            # Update running stats with exponential moving average
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * mu
            self.running_var  = self.momentum * self.running_var  + (1 - self.momentum) * var

            self._cache = {'z_hat': z_hat, 'mu': mu, 'var': var, 'z_in': z}
        else:
            z_hat = (z - self.running_mean) / np.sqrt(self.running_var + self.eps)

        return self.gamma * z_hat + self.beta

    def backward(self, grad_out, alpha):
        z_hat = self._cache['z_hat']
        var   = self._cache['var']
        z_in  = self._cache['z_in']
        mu    = self._cache['mu']
        N     = z_in.shape[0]

        d_gamma = (grad_out * z_hat).sum(axis=0, keepdims=True)
        d_beta  = grad_out.sum(axis=0, keepdims=True)

        d_zhat  = grad_out * self.gamma
        d_var   = (-0.5 * d_zhat * (z_in - mu) * (var + self.eps) ** (-1.5)).sum(axis=0, keepdims=True)
        d_mu    = (-d_zhat / np.sqrt(var + self.eps)).sum(axis=0, keepdims=True)
        d_z     = (d_zhat / np.sqrt(var + self.eps) +
                   2.0 * d_var * (z_in - mu) / N +
                   d_mu / N)

        # Update gamma and beta
        self.gamma -= alpha * d_gamma
        self.beta  -= alpha * d_beta
        return d_z

In [ ]:
# ── Dropout Layer ─────────────────────────────────────────────────────────────

class DropoutLayer:
    def __init__(self, drop_rate=0.3):
        assert 0.0 <= drop_rate < 1.0, "drop_rate must be in [0, 1)"
        self.drop_rate = drop_rate
        self._mask = None

    def forward(self, activations, training=True):
        if not training or self.drop_rate == 0.0:
            return activations
        # Bernoulli mask: 1 = keep, 0 = drop
        keep_prob = 1.0 - self.drop_rate
        self._mask = (np.random.rand(*activations.shape) < keep_prob).astype(np.float32)
        return activations * self._mask / keep_prob   # inverted scaling

    def backward(self, grad_upstream):
        if self._mask is None:
            return grad_upstream
        keep_prob = 1.0 - self.drop_rate
        return grad_upstream * self._mask / keep_prob

### Section 3 — The Network

In [ ]:
class RegularizedCardNet:
    def __init__(self, input_sz, hidden_szs, output_sz,
                 act_fn='relu', drop_p=0.2, l2_lam=0.001):
        dims = [input_sz] + hidden_szs + [output_sz]
        n_hidden = len(hidden_szs)

        self.linear_blocks = []
        self.bn_blocks      = []
        self.dropout_blocks = []
        self.act_fn_name    = act_fn

        for k in range(len(dims) - 1):
            is_last = (k == len(dims) - 2)
            # Linear with L2 on hidden layers only
            self.linear_blocks.append(
                LinearUnit(dims[k], dims[k+1], l2_lambda=0.0 if is_last else l2_lam)
            )
            if not is_last:
                self.bn_blocks.append(BatchNorm(dims[k+1]))
                self.dropout_blocks.append(DropoutLayer(drop_rate=drop_p))

    def forward(self, X, training=True):
        self._fwd_cache = []   # store (z, bn_z, act, drop_out) per hidden layer
        h = X
        n_hidden = len(self.bn_blocks)

        for k in range(n_hidden):
            z      = self.linear_blocks[k].forward(h)
            bn_z   = self.bn_blocks[k].forward(z, training=training)

            if self.act_fn_name == 'relu':
                act = rectify(bn_z)
            else:
                act = logistic(bn_z)

            drop_out = self.dropout_blocks[k].forward(act, training=training)
            self._fwd_cache.append((z, bn_z, act, drop_out))
            h = drop_out

        # Output layer — linear + softmax
        out_logits = self.linear_blocks[-1].forward(h)
        self._out_probs = stable_softmax(out_logits)
        self._out_logits = out_logits
        return self._out_probs

    def backward(self, targets, alpha, class_wts):
        # Gradient from weighted cross-entropy through softmax output
        g = weighted_cross_entropy_grad(self._out_probs, targets, class_wts)

        # Output linear layer backward
        g = self.linear_blocks[-1].backward(g, alpha)

        # Propagate back through hidden layers in reverse
        n_hidden = len(self.bn_blocks)
        for k in reversed(range(n_hidden)):
            z, bn_z, act, drop_out = self._fwd_cache[k]

            # Through dropout
            g = self.dropout_blocks[k].backward(g)

            # Through activation
            if self.act_fn_name == 'relu':
                g = rectify_prime(bn_z) * g
            else:
                g = logistic_prime(bn_z) * g

            # Through batch norm
            g = self.bn_blocks[k].backward(g, alpha)

            # Through linear
            g = self.linear_blocks[k].backward(g, alpha)

    def predict(self, X):
        probs = self.forward(X, training=False)
        return np.argmax(probs, axis=1)

### Section 4 — Training Loop

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

class TrainingLoop:
    def __init__(self, model, lr_init=0.05, total_epochs=30,
                 batch_sz=64, lr_decay=0.01):
        self.model        = model
        self.lr_init      = lr_init
        self.total_epochs = total_epochs
        self.batch_sz     = batch_sz
        self.lr_decay     = lr_decay

        self.loss_log = []
        self.acc_log  = []

    def run(self, feat_matrix, label_vec, class_wts):
        for ep in range(self.total_epochs):
            cur_lr = self.lr_init / (1.0 + self.lr_decay * ep)

            # Shuffle each epoch
            order = np.random.permutation(feat_matrix.shape[0])
            Xp, yp = feat_matrix[order], label_vec[order]

            # Mini-batch updates
            for b0 in range(0, Xp.shape[0], self.batch_sz):
                xb = Xp[b0 : b0 + self.batch_sz]
                yb = yp[b0 : b0 + self.batch_sz]
                self.model.forward(xb, training=True)
                self.model.backward(yb, cur_lr, class_wts)

            # End-of-epoch metrics (inference mode: dropout off, BN uses running stats)
            ep_probs = self.model.forward(feat_matrix, training=False)
            ep_loss  = weighted_cross_entropy(ep_probs, label_vec, class_wts)
            ep_preds = np.argmax(ep_probs, axis=1)
            ep_acc   = np.mean(ep_preds == label_vec)

            self.loss_log.append(ep_loss)
            self.acc_log.append(ep_acc)

            print(f"[Epoch {ep+1:02d}]  lr={cur_lr:.5f}  "
                  f"loss={ep_loss:.4f}  acc={ep_acc:.4f}")

    def score(self, X_eval, y_eval):
        preds = self.model.predict(X_eval)
        acc   = np.mean(preds == y_eval)
        print(f"\n>>> Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
        return preds

    def show_curves(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

        ax1.plot(range(1, len(self.acc_log)+1), self.acc_log,
                 color='mediumseagreen', linewidth=2)
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training Accuracy')
        ax1.set_title('Learning Curve — Accuracy'); ax1.grid(alpha=0.4)

        ax2.plot(range(1, len(self.loss_log)+1), self.loss_log,
                 color='indianred', linewidth=2)
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('Weighted Cross-Entropy Loss')
        ax2.set_title('Learning Curve — Loss'); ax2.grid(alpha=0.4)

        plt.suptitle('Training Diagnostics', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def show_confusion(self, y_true, y_pred, title='Confusion Matrix'):
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(10, 8))
        ConfusionMatrixDisplay(cm).plot(ax=ax, cmap=plt.cm.YlOrRd)
        ax.set_title(title, fontsize=13)
        plt.tight_layout()
        plt.show()

### Section 5 — Dataset

Link: https://archive.ics.uci.edu/dataset/158/poker+hand


In [ ]:
import os, urllib.request, zipfile

STORAGE = "raw_cards"
os.makedirs(STORAGE, exist_ok=True)

SRC_URL  = "https://archive.ics.uci.edu/static/public/158/poker+hand.zip"
arc_file = os.path.join(STORAGE, "ph_archive.zip")

print("Fetching dataset...")
urllib.request.urlretrieve(SRC_URL, arc_file)

with zipfile.ZipFile(arc_file) as zf:
    zf.extractall(STORAGE)

print("Extracted:", sorted(os.listdir(STORAGE)))

In [ ]:
def ingest_csv(fpath):
    mat = np.loadtxt(fpath, delimiter=',', dtype=np.int32)
    return mat[:, :10], mat[:, 10]

feat_tr, lbl_tr = ingest_csv(f"{STORAGE}/poker-hand-training-true.data")
feat_te, lbl_te = ingest_csv(f"{STORAGE}/poker-hand-testing.data")

print(f"Train → features: {feat_tr.shape}  labels: {lbl_tr.shape}")
print(f"Test  → features: {feat_te.shape}  labels: {lbl_te.shape}")

### Section 6 — Exploratory Analysis

In [ ]:
import pandas as pd
import seaborn as sns

HAND_LABELS = [
    'Nothing', 'One Pair', 'Two Pairs', 'Three of a Kind',
    'Straight', 'Flush', 'Full House', 'Four of a Kind',
    'Straight Flush', 'Royal Flush'
]

hdr = [f"S{c}" for c in range(1,6)] + [f"R{c}" for c in range(1,6)]
# Reorder to match original: S1,R1,S2,R2...
hdr2 = []
for c in range(1, 6):
    hdr2 += [f"S{c}", f"R{c}"]
hdr2 += ["hand"]

card_df = pd.read_csv(f"{STORAGE}/poker-hand-training-true.data", names=hdr2)
print(card_df.describe())

In [ ]:
# Class imbalance visualization
hand_freq = card_df['hand'].value_counts().sort_index()
total_hands = len(card_df)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
palette = sns.color_palette('magma', 10)
axes[0].barh(range(10), hand_freq.values, color=palette)
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(HAND_LABELS)
axes[0].set_xlabel('Sample Count')
axes[0].set_title('Class Distribution (log scale)')
axes[0].set_xscale('log')

# Pie chart — top 4 classes only (others too small to show)
top4 = hand_freq.nlargest(4)
others = total_hands - top4.sum()
pie_vals  = list(top4.values) + [others]
pie_lbls  = [HAND_LABELS[i] for i in top4.index] + ['Rare (5–9)']
axes[1].pie(pie_vals, labels=pie_lbls, autopct='%1.1f%%',
            colors=sns.color_palette('pastel', 5))
axes[1].set_title('Class Proportions')

plt.suptitle('Poker Hand Dataset — Class Imbalance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nDetailed class breakdown:")
for i, name in enumerate(HAND_LABELS):
    cnt = hand_freq.get(i, 0)
    print(f"  [{i}] {name:20s}: {cnt:6d}  ({cnt/total_hands*100:.3f}%)")

In [ ]:
# Suit and rank distributions
rank_cols_list = [f"R{c}" for c in range(1, 6)]
suit_cols_list = [f"S{c}" for c in range(1, 6)]

all_ranks_flat = card_df[rank_cols_list].values.flatten()
all_suits_flat = card_df[suit_cols_list].values.flatten()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, 14),
            [np.sum(all_ranks_flat == r) for r in range(1, 14)],
            color=sns.color_palette('Blues_d', 13))
axes[0].set_xticks(range(1, 14))
axes[0].set_xticklabels(['A','2','3','4','5','6','7','8','9','10','J','Q','K'])
axes[0].set_title('Rank Frequency'); axes[0].set_ylabel('Count')

suit_names = ['Clubs', 'Diamonds', 'Hearts', 'Spades']
axes[1].bar(suit_names,
            [np.sum(all_suits_flat == s) for s in range(1, 5)],
            color=['#5b5b5b', '#e05c5c', '#e05c5c', '#5b5b5b'])
axes[1].set_title('Suit Frequency'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### Section 7 — Feature Engineering


In [ ]:
def binarize_hand(raw_matrix):
    N = raw_matrix.shape[0]
    binary_out = np.zeros((N, 85), dtype=np.float32)

    for card_no in range(5):
        suit_idx = 2 * card_no
        rank_idx = 2 * card_no + 1

        sv = raw_matrix[:, suit_idx] - 1    # suits  1→4 become 0→3
        rv = raw_matrix[:, rank_idx] - 1    # ranks  1→13 become 0→12

        blk = card_no * 17
        for row in range(N):
            binary_out[row, blk + sv[row]]       = 1.0   # suit bit
            binary_out[row, blk + 4 + rv[row]]   = 1.0   # rank bit

    return binary_out

Xtr_bin = binarize_hand(feat_tr)
Xte_bin = binarize_hand(feat_te)

print(f"Binary train: {Xtr_bin.shape}")
print(f"Binary test:  {Xte_bin.shape}")
print(f"\nSample raw  → {feat_tr[0]}")
print(f"Sample bin  → {Xtr_bin[0][:20]} ...")

In [ ]:
# Compute class weights from training labels
K = 10
cw = compute_class_weights(lbl_tr, K)

print("Class weights (higher = rarer class):")
for i, (name, w) in enumerate(zip(HAND_LABELS, cw)):
    print(f"  [{i}] {name:20s}: weight = {w:.4f}")

### Section 8 — Training the Regularized Model

In [ ]:
INPUT_NODES  = 85
HIDDEN_NODES = [64, 64]
OUTPUT_NODES = 10

INIT_LR   = 0.05
N_EPOCHS  = 30
BSIZE     = 64
DECAY_RT  = 0.01
DROP_RATE = 0.2
L2_COEF   = 0.001

card_net = RegularizedCardNet(
    input_sz  = INPUT_NODES,
    hidden_szs= HIDDEN_NODES,
    output_sz = OUTPUT_NODES,
    act_fn    = 'relu',
    drop_p    = DROP_RATE,
    l2_lam    = L2_COEF
)

loop = TrainingLoop(
    model        = card_net,
    lr_init      = INIT_LR,
    total_epochs = N_EPOCHS,
    batch_sz     = BSIZE,
    lr_decay     = DECAY_RT
)

loop.run(Xtr_bin, lbl_tr, class_wts=cw)

In [ ]:
# Test evaluation
final_preds = loop.score(Xte_bin, lbl_te)

In [ ]:
loop.show_curves()

In [ ]:
loop.show_confusion(lbl_te, final_preds, title='Confusion Matrix — Regularized Card Net')

### Section 9 — Ablation: Effect of Regularization Components



In [ ]:
ablation_configs = [
    {'label': 'Full model (BN + Dropout + L2 + CW)', 'drop_p': 0.2, 'l2': 0.001, 'weighted': True},
    {'label': 'No Dropout',                          'drop_p': 0.0, 'l2': 0.001, 'weighted': True},
    {'label': 'No L2 decay',                         'drop_p': 0.2, 'l2': 0.0,   'weighted': True},
    {'label': 'No class weighting',                  'drop_p': 0.2, 'l2': 0.001, 'weighted': False},
]

uniform_wts = np.ones(K, dtype=np.float32)
ablation_results = {}

for cfg in ablation_configs:
    print(f"\n--- {cfg['label']} ---")
    net_ab = RegularizedCardNet(INPUT_NODES, HIDDEN_NODES, OUTPUT_NODES,
                                drop_p=cfg['drop_p'], l2_lam=cfg['l2'])
    lp_ab  = TrainingLoop(net_ab, lr_init=INIT_LR, total_epochs=N_EPOCHS,
                          batch_sz=BSIZE, lr_decay=DECAY_RT)
    wts_to_use = cw if cfg['weighted'] else uniform_wts
    lp_ab.run(Xtr_bin, lbl_tr, class_wts=wts_to_use)
    ab_preds = lp_ab.score(Xte_bin, lbl_te)
    ablation_results[cfg['label']] = np.mean(ab_preds == lbl_te)

In [ ]:
# Summary bar chart
labels_ab = list(ablation_results.keys())
accs_ab   = list(ablation_results.values())

plt.figure(figsize=(10, 4))
bars = plt.barh(labels_ab, accs_ab, color=sns.color_palette('coolwarm', len(labels_ab)))
plt.xlabel('Test Accuracy')
plt.title('Ablation Study — Contribution of Each Regularization Component')
plt.xlim(0.8, 1.0)
for bar, acc in zip(bars, accs_ab):
    plt.text(acc + 0.001, bar.get_y() + bar.get_height()/2,
             f'{acc:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### Section 10 — Sklearn Baseline

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

ref_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 64),
    activation='relu',
    solver='sgd',
    batch_size=64,
    learning_rate_init=0.1,
    max_iter=30,
    random_state=42
)
ref_mlp.fit(Xtr_bin, lbl_tr)
ref_preds = ref_mlp.predict(Xte_bin)
ref_acc   = accuracy_score(lbl_te, ref_preds)

our_acc = np.mean(final_preds == lbl_te)
print(f"Our regularized model : {our_acc:.4f}")
print(f"Sklearn MLPClassifier  : {ref_acc:.4f}")

In [ ]:
# Classification report for sklearn baseline
print(classification_report(lbl_te, ref_preds, target_names=HAND_LABELS, zero_division=0))

In [ ]:
cm_ref = confusion_matrix(lbl_te, ref_preds)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm_ref).plot(ax=ax, cmap=plt.cm.Blues)
ax.set_title('Confusion Matrix — Sklearn Baseline')
plt.tight_layout()
plt.show()